# Kết quả dự án nhận diện cây dược liệu v0.3
Huỳnh Gia Huy - 2474802010135 - 03/11/2026.

Notebook đọc nhật ký đã lưu, tính lại chỉ số từ dự đoán OOF và kiểm tra mô hình. Run All không huấn luyện hoặc dự đoán lại test lịch sử.

In [1]:
from pathlib import Path
import json
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "results/phase3/nested_evaluation.json").exists())
R = ROOT / "results/phase3"
evaluation = json.loads((R / "nested_evaluation.json").read_text())
predictions = pd.read_csv(R / "nested_predictions.csv")
metadata = json.loads((ROOT / "models/phase3_metadata.json").read_text())
print("Số dự đoán outer OOF:", len(predictions))
print("Phạm vi:", evaluation["scope"])

Số dự đoán outer OOF: 778
Phạm vi: nested_grouped_cv_on_previous_development


## Trước và sau tối ưu
Đây là kết quả nested CV trên development cũ. Chính sách chọn cấu hình 4 tham số được chấm ở outer-validation; mô hình refit cuối chưa có một tập test mới.

In [2]:
comparison = []
for model in ["baseline", "selected4"]:
    f1 = f1_score(predictions.label, predictions[model], average="macro")
    acc = accuracy_score(predictions.label, predictions[model])
    assert abs(f1 - evaluation["metrics"][model]["macro_f1"]) < 1e-12
    comparison.append({"model": model, "Macro-F1 %": 100*f1, "Accuracy %": 100*acc})
print(pd.DataFrame(comparison).to_string(index=False))

    model  Macro-F1 %  Accuracy %
 baseline   86.168921   86.118252
selected4   88.061359   88.046272


## Đối chứng 3 và 4 tham số
Mean/SD giữa seed không phải điểm của các bộ dữ liệu độc lập. Chính sách selected4 chọn qua hai phương pháp/năm seed nên có tổng ngân sách khác một search của một seed. Không kết luận bốn tham số luôn tốt hơn ba.

In [3]:
rows = [{"search": k, "mean OOF %": 100*v["mean_oof_macro_f1"], "SD điểm %": 100*v["sd_between_seeds"]}
        for k,v in evaluation["search_aggregate"].items()]
print(pd.DataFrame(rows).to_string(index=False))
log = [json.loads(line) for line in (R/"trials.jsonl").read_text().splitlines()]
assert len(log) == 3600
print("HPO trials:", len(log), "Inner CV fit:", 3*len(log))

 search  mean OOF %  SD điểm %
random3   87.796482   0.422888
random4   86.359612   2.436542
   tpe4   87.294734   0.827477
HPO trials: 3600 Inner CV fit: 10800


## Ma trận nhầm lẫn và chỉ số từng loài

In [4]:
labels = evaluation["metrics"]["selected4"]["labels"]
matrix = pd.DataFrame(evaluation["metrics"]["selected4"]["confusion_matrix"], index=labels, columns=labels)
print(matrix.to_string())
print(pd.DataFrame(evaluation["metrics"]["selected4"]["classification_report"]).T.to_string())

                     Centella Asiatica  Justicia Adhatoda  Kalanchoe Pinnata  Mikania Micrantha  Phyllanthus Emblica  Terminalia Arjuna
Centella Asiatica                  119                  7                  0                  3                    3                  0
Justicia Adhatoda                    8                107                  0                  4                    5                  6
Kalanchoe Pinnata                    1                  0                124                  3                    1                  0
Mikania Micrantha                    6                  3                  5                109                    2                  3
Phyllanthus Emblica                  6                  1                  0                  5                  111                  2
Terminalia Arjuna                    1                  7                  0                  9                    2                115
                     precision    recall  f1-sco

## Cấu hình cuối và nguồn gốc
Giữ kết quả test lịch sử riêng. Điểm 88,71% của giai đoạn 2 không so trực tiếp với điểm OOF mới. class_weight=None là giá trị được tìm ra, không phải bỏ qua tham số thứ tư.

In [5]:
import hashlib
model_path = ROOT / "models" / metadata["model_file"]
assert hashlib.sha256(model_path.read_bytes()).hexdigest() == metadata["model_sha256"]
print(json.dumps({k:metadata[k] for k in ["params", "selected_search", "training_images", "external_validation_completed"]}, indent=2))
old = json.loads((ROOT/"results/phase2/evaluation/metrics.json").read_text())
print("Macro-F1 test lịch sử Random3:", old["random"]["test_macro_f1"])
print("Không chấm lại test lịch sử trong notebook này.")

{
  "params": {
    "kernel": "rbf",
    "C": 6.602593199343702,
    "gamma": 0.005818740333111208,
    "class_weight": null
  },
  "selected_search": "tpe4",
  "training_images": 778,
  "external_validation_completed": false
}
Macro-F1 test lịch sử Random3: 0.8870532222751879
Không chấm lại test lịch sử trong notebook này.


## Dữ liệu Việt Nam và giới hạn
Biểu diễn texture128 đã được chọn trên development trong lịch sử. Nhóm là heuristic ảnh trùng, chưa phải ID cây. Bộ kiểm tra hiện chặn dữ liệu Việt Nam chưa xác minh; mô hình vẫn là sáu lớp ảnh lá, chưa huấn luyện nhiều bộ phận/học sâu hoặc phát hiện loài lạ.

In [6]:
readiness = json.loads((ROOT/"research/vietnam/readiness_v0_1.json").read_text())
assert readiness["ready_for_training"] is False
print(json.dumps(readiness, ensure_ascii=False, indent=2))

{
  "valid": true,
  "ready_for_training": false,
  "n_records": 69,
  "n_train_keep": 0,
  "n_blocked": 69,
  "verified_species_counts": {},
  "multi_organ_observations": 0,
  "errors": [],
  "warnings": [
    "No images have completed botanical and rights review",
    "Image bytes not checked; pass --images-root before training release"
  ],
  "note": "Passing validation checks structure and recorded evidence, not botanical truth or sufficient statistical power."
}
